# Bài học 15: Prompt, lập kế hoạch, và học hỏi với Claude Code

**Thời gian:** ~40 phút | **Chi phí:** $0 (không gọi API)

Bạn đã biết Claude Code là gì (Bài 13) và cách sử dụng nó (Bài 14). Bài học này dạy các **kỹ năng** tạo nên sự khác biệt giữa "Claude Code tàm tạm" và "Claude Code làm chính xác những gì tôi cần."

Ba kỹ năng:
1. **Viết prompt** — cách viết chỉ dẫn để có kết quả đúng
2. **Lập kế hoạch** — cách chia nhỏ nhiệm vụ phức tạp trước khi xây dựng
3. **Học hỏi** — cách hiểu code lạ và xử lý lỗi

## Quy trình phát triển

```
1. Tìm hiểu → 2. Lên kế hoạch → 3. Triển khai → 4. Kiểm tra → 5. Lặp lại
```

Quy trình này ánh xạ trực tiếp với những gì bạn đã biết:

| Bước | Bạn làm gì | Tương đương agent pipeline |
|------|-----------|----------------------------|
| **Tìm hiểu** | Đọc code, hiểu trạng thái hiện tại | Research Agent tìm kiếm web |
| **Lên kế hoạch** | Thiết kế cần thay đổi gì, theo thứ tự nào | Outline Agent tạo cấu trúc |
| **Triển khai** | Claude Code viết code | Content Writer viết bài |
| **Kiểm tra** | Bạn review — có đúng ý bạn không? | Con người review bài viết |
| **Lặp lại** | Sửa chỗ sai, cải thiện chỗ gần đúng | Chỉnh sửa và chạy lại |

**Điểm mấu chốt**: Bước 1, 2, 4, và 5 là việc của BẠN. Bước 3 là việc của Claude Code. Bạn dành nhiều thời gian suy nghĩ và review hơn là viết code.

## Kỹ năng 1: Viết prompt cho thay đổi code

Các nguyên tắc từ Bài học 6 (Prompt và Ngữ cảnh) áp dụng trực tiếp ở đây. Nhưng prompt cho code cần các yếu tố cụ thể:

### 4 yếu tố của một prompt tốt:

1. **WHERE** — file nào cần chỉnh sửa
2. **WHAT** — hành vi cần thêm hoặc thay đổi
3. **HOW** — ràng buộc, pattern cần tuân theo
4. **VERIFY** — cách xác nhận kết quả đúng

### Ví dụ từ dự án:

**Tệ**: "Làm writer tốt hơn"
**Tốt**: "Trong `output/backend/agents/content_writer.py`, thêm chỉ dẫn rằng bài viết phải có ít nhất 3 liên kết nội bộ đến các bài viết khác. Dùng `list_all_articles()` để tìm chủ đề bài viết hiện có. Sau khi triển khai, tạo một bài viết thử để xác nhận liên kết được thêm."

**Tệ**: "Sửa storage đi"
**Tốt**: "Trong `output/backend/tools/storage.py`, hàm `save_article` không kiểm tra `topic` có rỗng không. Thêm kiểm tra ở đầu hàm, raise ValueError nếu topic rỗng hoặc chỉ có khoảng trắng."

In [ ]:
# Phân tích điều gì làm prompt hiệu quả
# Chấm điểm mỗi prompt theo 4 yếu tố: WHERE, WHAT, HOW, VERIFY

prompts = [
    {
        "prompt": "Fix the agents",
        "where": False,  # No file specified
        "what": False,   # "Fix" what exactly?
        "how": False,    # No constraints
        "verify": False, # No way to check
        "score": 0,
    },
    {
        "prompt": "In content_writer.py, make articles longer",
        "where": True,   # File specified
        "what": False,   # "Longer" is vague
        "how": False,    # How much longer? What sections?
        "verify": False, # No verification method
        "score": 1,
    },
    {
        "prompt": "In output/backend/agents/content_writer.py, update instructions to require minimum 2000 words. Add a note that each H2 section should be at least 200 words. Run the writer on a test topic to verify.",
        "where": True,
        "what": True,    # Minimum 2000 words, 200 per section
        "how": True,     # Specific word counts
        "verify": True,  # Test on a topic
        "score": 4,
    },
]

for p in prompts:
    elements = sum([p["where"], p["what"], p["how"], p["verify"]])
    print(f"Score: {elements}/4")
    if len(p['prompt']) > 80:
        print(f"  Prompt: \"{p['prompt'][:80]}...\"")
    else:
        print(f"  Prompt: \"{p['prompt']}\"")
    missing = []
    if not p["where"]: missing.append("WHERE (no file)")
    if not p["what"]: missing.append("WHAT (vague goal)")
    if not p["how"]: missing.append("HOW (no constraints)")
    if not p["verify"]: missing.append("VERIFY (no test)")
    if missing:
        print(f"  Missing: {', '.join(missing)}")
    print()

## Kỹ năng 2: Lập kế hoạch trước khi xây dựng

### Khi nào dùng plan mode

Plan mode (`Shift+Tab` trong Claude Code) yêu cầu Claude Code **khám phá và thiết kế** trước khi thay đổi. Dùng khi:

- Bạn thay đổi **hơn 2 file**
- Bạn **không chắc** về cách tiếp cận tốt nhất
- Nhiệm vụ **phức tạp** (tính năng mới, thay đổi kiến trúc)
- Bạn muốn **hiểu** code hiện tại trước

### Plan mode hoạt động thế nào

1. Bạn mô tả điều bạn muốn
2. Claude Code đọc các file liên quan, khám phá codebase
3. Nó trình bày **kế hoạch** — file nào cần thay đổi, thay đổi gì, theo thứ tự nào
4. Bạn **review** kế hoạch — có hợp lý không? Có tuân theo pattern hiện có không?
5. Bạn duyệt → Claude Code triển khai

### Ví dụ: Thêm agent Proofreader

Không dùng plan mode:
> "Add a proofreader agent"
> Claude Code có thể tạo file không theo pattern hiện có, quên đăng ký trong \_\_init\_\_.py, hoặc bỏ sót tích hợp vào team.py.

Dùng plan mode:
> "I want to add a Proofreader agent. Use plan mode to explore the existing agent files and design the approach first."
> Claude Code đọc content_writer.py, image_finder.py, aio_analyzer.py, team.py, \_\_init\_\_.py, và trình bày:
> - Tạo `agents/proofreader.py` theo pattern từ content_writer.py
> - Thêm export vào `agents/__init__.py`
> - Thêm làm member trong `agents/team.py`
> - Dùng Claude Sonnet (giống các agent khác)
> Bạn review, duyệt, rồi nó triển khai.

### Review kế hoạch

Khi Claude Code trình bày kế hoạch, hãy kiểm tra:
- Có sửa **đúng file** không? (không có file ngoài dự kiến)
- Có tuân theo **pattern hiện có** không? (cùng model, cùng tool style)
- **Thứ tự** có đúng không? (tạo trước khi import, định nghĩa trước khi dùng)
- Có **thiếu** gì không? (quên cập nhật \_\_init\_\_.py?)

### Duyệt một phần — "Đúng rồi, nhưng..."

Bạn không phải lúc nào cũng duyệt hoặc từ chối toàn bộ kế hoạch. Phản hồi phổ biến nhất là **duyệt một phần**: "Đúng rồi, nhưng sửa chỗ này."

Ví dụ — Claude Code đề xuất thêm Image Finder:

> **Kế hoạch của Claude Code:**
> 1. Tạo `agents/image_finder.py` tích hợp Unsplash API
> 2. Thêm `tools/unsplash.py` toolkit
> 3. Đăng ký trong team.py

> **Phản hồi của bạn:** "Cấu trúc đúng rồi, nhưng dùng DataForSEO Images thay vì Unsplash — mình đã có sẵn credentials DataForSEO. Và làm nó optional: nếu không có API key, agent không nên được tạo."

Claude Code sửa lại kế hoạch. Bạn kiểm tra lần nữa. Cách trao đổi qua lại này **nhanh hơn** viết prompt hoàn hảo ngay từ đầu, vì bạn đang phản hồi thứ cụ thể thay vì cố gắng chỉ định mọi thứ từ trí nhớ.

**Khi nào nên phản đối:**
- Model choice không khớp với các agent hiện có (Bài 7)
- Nó tạo pattern mới thay vì tuân theo pattern hiện có (Bài 19)
- Nó over-engineer — thêm phức tạp bạn không yêu cầu
- Nó bỏ sót dependency (quên `__init__.py`, quên team.py)

## Kỹ năng 3: Học những khái niệm mới

Các bài tiếp theo sẽ cho bạn xem code production với những khái niệm chưa học: FastAPI, React, SSE streaming, CORS. **Đây là bình thường.** Bạn không cần biết mọi thứ — bạn cần biết cách hỏi.

### Chiến lược 1: "Giải thích như tôi không phải developer"

Khi thấy code bạn không hiểu:
> "Explain what this code does in `serve.py` lines 15-30. I'm not a developer — explain the PURPOSE, not the syntax."

Claude Code sẽ giải thích logic nghiệp vụ, không phải chi tiết Python.

### Chiến lược 2: "Nếu tôi thay đổi cái này thì gì sẽ hỏng?"

Trước khi sửa code lạ:
> "I want to change the API route from `/api/articles` to `/api/posts`. What other files reference this path? What would break?"

Claude Code truy vết tất cả dependencies.

### Chiến lược 3: "Cho tôi xem phiên bản đơn giản hơn"

Khi code quá phức tạp:
> "The `serve.py` file uses AgentOS which is complex. Can you show me a simplified version (10 lines max) that demonstrates what it does conceptually?"

### Chiến lược 4: "Lỗi này nghĩa là gì?"

Khi chương trình lỗi:
> "I got this error when running `python output/backend/serve.py`:
> ```
> ModuleNotFoundError: No module named 'agno.os'
> ```
> What does this mean and how do I fix it?"

**Đừng cố tự hiểu lỗi.** Paste toàn bộ thông báo lỗi vào Claude Code và hỏi chuyện gì đã xảy ra.

In [ ]:
# Đây là code thực tế từ serve.py (đã đơn giản hóa)
# Đừng lo về việc hiểu từng dòng
# Tập trung vào xác định code LÀM GÌ, không phải LÀM THẾ NÀO

production_code = '''
from agno.os import AgentOS

app = FastAPI()

@app.get("/api/articles")
def list_articles():
    return list_all_articles()

@app.post("/api/articles/{article_id}")  
def update_article(article_id: str, body: dict):
    return update_article_content(article_id, body["content"])

AgentOS(teams=[team], base_app=app).start(port=7777)
'''

print("Code này làm gì (nhìn tổng thể):")
print("1. Tạo web server (FastAPI)")
print("2. Thêm 2 API route cho bài viết (liệt kê + cập nhật)")
print("3. Kết nối agent team với web server (AgentOS)")
print("4. Khởi chạy tất cả trên port 7777")
print()
print("Những gì bạn KHÔNG CẦN biết:")
print("- @app.get nghĩa là gì (đó là decorator — khái niệm Python)")
print("- FastAPI routing hoạt động thế nào bên trong")
print("- AgentOS làm gì ở bên dưới")
print()
print("Những gì bạn CẦN biết:")
print("- File này là điểm vào của web backend")
print("- Nó kết nối agent với giao diện web")
print("- Port 7777 là nơi server chạy")
print("- Bài viết có thể được liệt kê và cập nhật qua web request")

## Debug với Claude Code

Debug là lúc pattern hội thoại quan trọng nhất. Đừng chỉ paste lỗi rồi chấp nhận bản sửa đầu tiên — hãy **hiểu chuyện gì đã xảy ra** để cùng bug không quay lại.

Đây là cuộc hội thoại debug thực tế:

> **Bạn:** "Tôi chạy `python output/backend/serve.py` và bị lỗi:
> ```
> ImportError: cannot import name 'proofreader' from 'agents'
> ```"

> **Claude Code:** "Import thất bại vì `proofreader` chưa được export từ `agents/__init__.py`. Tôi sẽ thêm dòng export."

> **Bạn:** "Khoan — trước khi sửa, tại sao nó chưa có ở đó? Mình quên thêm hay có vấn đề khác?"

> **Claude Code:** *đọc \_\_init\_\_.py, đọc proofreader.py:* "File `agents/proofreader.py` đã tồn tại nhưng `agents/__init__.py` chưa bao giờ được cập nhật. Khi tạo agent ở phiên trước, bước cập nhật `__init__.py` bị bỏ sót."

> **Bạn:** "OK, sửa đi. Và kiểm tra xem có chỗ nào khác mà proofreader cần được đăng ký mà chưa có không — ví dụ team.py."

**Thói quen then chốt:** Hỏi "tại sao lỗi này xảy ra?" trước khi nói "sửa đi." Điều này biến debug từ trò đập chuột chũi thành quá trình hiểu thật sự.

Debug theo pattern đơn giản:

1. **Thấy lỗi** — copy TOÀN BỘ thông báo lỗi (không chỉ dòng cuối)
2. **Hỏi Claude Code** — paste lỗi và hỏi "What went wrong and how do I fix it?"
3. **Hiểu cách sửa** — hỏi "Why did this happen?" nếu bạn muốn học
4. **Xác nhận** — chạy code lại để xác nhận đã sửa xong

### Các lỗi thường gặp:

| Lỗi | Nghĩa là gì | Prompt cho Claude Code |
|------|------------|------------------------|
| `ModuleNotFoundError` | Package chưa được cài | "I got ModuleNotFoundError for X. Install it and verify." |
| `ImportError` | Đường dẫn import sai | "Fix this ImportError in [file]. The module moved." |
| `KeyError` | Dictionary thiếu key | "This KeyError in [file] says key 'X' is missing. Check where it's set." |
| `FileNotFoundError` | Đường dẫn file sai | "This file path in [file] is wrong. Find the correct path." |
| `ConnectionError` | API/server không truy cập được | "The server can't connect to [service]. Check if it's running." |

**Kỹ năng then chốt**: Đừng cố debug một mình. Khi thấy lỗi bạn không hiểu, paste ngay vào Claude Code. Đó chính là lý do nó tồn tại.

## Tổng kết Bài 15

### Ba kỹ năng cho phát triển với AI:

| Kỹ năng | Điểm chính |
|---------|----------|
| **Viết prompt** | Dùng 4 yếu tố: WHERE, WHAT, HOW, VERIFY |
| **Lập kế hoạch** | Dùng plan mode cho nhiệm vụ phức tạp. Review kế hoạch trước khi duyệt. |
| **Học hỏi** | Nhờ Claude Code giải thích code lạ. Paste toàn bộ lỗi. |

### Quy trình 5 bước:
```
Tìm hiểu → Lên kế hoạch → Triển khai   → Kiểm tra → Lặp lại
   BẠN         BẠN        CLAUDE CODE      BẠN        BẠN
```

Giá trị của bạn nằm ở bước 1, 2, 4, và 5. Claude Code lo bước 3.

**Bài tiếp theo:** Xây dựng Content Writer — lần đầu xem code production agent, qua lăng kính "bạn sẽ chỉ đạo Claude Code xây dựng cái này như thế nào?"

## Bài tập

Viết prompt cho Claude Code cho mỗi tình huống. Chấm điểm mỗi prompt theo 4 yếu tố (WHERE, WHAT, HOW, VERIFY).

In [ ]:
# Bài tập: Viết 5 prompt với độ khó tăng dần
# Điền vào chỗ trống để tạo prompt đạt 4/4

# 1. DỄ: Thay đổi phong cách writer
easy = """
In output/backend/agents/___,
change the writing tone from formal to ___.
Keep all other instructions about ___ and word count unchanged.
Create a test article to verify the ___ changed.
"""

# 2. TRUNG BÌNH: Thêm trường vào storage
medium = """
In output/backend/tools/___,
add a ___ field to the article metadata in save_article.
Calculate it as word_count / ___ (average reading speed).
Update ___ to display this field when listing articles.
Test by saving an article and checking the new field appears.
"""

# 3. KHÓ: Thêm agent mới
hard = """
Use plan mode first. I want to add a ___ agent that checks
articles for grammar and spelling errors.
Follow the pattern in output/backend/agents/___.py.
Register it in output/backend/agents/___.
Add it as a member in output/backend/agents/___.
Use Claude ___ (same model as other agents).
Test by running it on an existing article.
"""

# 4. DEBUG: Sửa lỗi
debug = """
When I run `python output/backend/serve.py`, I get:
```
ImportError: cannot import name '___' from 'agents'
```
Find where this import is used, check what's actually
exported from agents/__init__.py, and fix the ___.
"""

# 5. HỌC HỎI: Hiểu code lạ
learn = """
Explain what output/backend/___ does.
I'm not a developer \u2014 explain the PURPOSE of each section,
not the Python syntax. Focus on:
- What does this file do in the overall system?
- What would break if we deleted it?
- How does it connect to the ___ files?
"""

for name, prompt in [("Dễ", easy), ("Trung bình", medium), ("Khó", hard), ("Debug", debug), ("Học hỏi", learn)]:
    print(f"=== {name} ===")
    print(prompt)

<details>
<summary>Bấm để xem đáp án</summary>

**1. Dễ** (thay đổi phong cách writer):
- `content_writer.py`
- `casual and conversational`
- `SEO`
- `tone`

**2. Trung bình** (thêm trường vào storage):
- `storage.py`
- `reading_time_minutes`
- `200`
- `list_all_articles`

**3. Khó** (thêm agent mới):
- `proofreader`
- `content_writer`
- `__init__.py`
- `team.py`
- `Sonnet`

**4. Debug** (sửa lỗi):
- Tên import không tồn tại trong `agents/__init__.py`
- `mismatch` giữa import và những gì thực sự được export

**5. Học hỏi** (hiểu code lạ):
- `serve.py`
- `agents/`

</details>